# 下载数据集
- 下载cluneer数据集
- 下载corpus数据集

In [1]:
import os
from pathlib import Path

from datasets import load_dataset, DatasetDict, load_from_disk


# 所有数据集统一下载到当前目录下的 data 文件夹
DATA_DIR = Path.cwd() / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
!pip install -q seqeval pytorch-crf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [3]:
def download_dataset(repo_id: str, subdir: str, base_dir: Path = DATA_DIR,
                     force_download: bool = False) -> DatasetDict:
    """从 HuggingFace 下载数据集并保存到 data/<subdir> 下。

    每个分片同时保存为：
      - Arrow 格式（save_to_disk，加载快、保留类型信息）
      - JSON 格式（如 train.json，方便人工查看与跨工具调取）

    若本地已存在且未指定 force_download，则直接从磁盘加载，避免重复下载。

    Args:
        repo_id: HuggingFace 上的数据集仓库 id，如 "xusenlin/clue-ner"。
        subdir: 保存到 data 下的子文件夹名，如 "cluner"、"corpus"。
        base_dir: 数据根目录，默认为 DATA_DIR。
        force_download: 为 True 时即使本地已存在也重新下载。

    Returns:
        加载后的 DatasetDict。
    """
    target_dir = base_dir / subdir

    # 本地已有缓存则直接加载
    if target_dir.exists() and not force_download:
        print(f"[{subdir}] 已存在，直接从本地加载：{target_dir}")
        return load_from_disk(str(target_dir))

    print(f"[{subdir}] 正在从 HuggingFace 下载：{repo_id}")
    dataset = load_dataset(repo_id)

    target_dir.mkdir(parents=True, exist_ok=True)
    # 1. 保存为 Arrow 格式（整体 DatasetDict）
    dataset.save_to_disk(str(target_dir))

    # 2. 每个分片导出一份 JSON，如 data/cluner/train.json
    for split_name, split_data in dataset.items():
        json_path = target_dir / f"{split_name}.json"
        # force_ascii=False 保留中文；lines=True 为每行一条记录的 JSONL，方便流式读取
        split_data.to_json(str(json_path), force_ascii=False, lines=True)
        print(f"[{subdir}] 已导出 JSON：{json_path}")

    print(f"[{subdir}] 下载完成并保存至：{target_dir}，分片：{list(dataset.keys())}")
    return dataset

In [4]:
# 需要下载的数据集配置：repo_id -> data 下的子文件夹
DATASETS = {
    "xusenlin/clue-ner": "cluner",          # CLUE 命名实体识别数据集
    "lansinuote/peoples-daily-ner": "corpus",  # 人民日报命名实体识别数据集
}

# 批量下载所有配置好的数据集
datasets = {
    subdir: download_dataset(repo_id, subdir)
    for repo_id, subdir in DATASETS.items()
}

ds_clue = datasets["cluner"]
ds_corpus = datasets["corpus"]


def get_common_splits(dataset_dict: DatasetDict):
    """提取数据集的 train / validation / test 三个常用分片。"""
    return (
        dataset_dict.get("train"),
        dataset_dict.get("validation"),
        dataset_dict.get("test"),
    )


# CLUE 的训练、验证和测试
clue_train_ds, clue_validation_ds, clue_test_ds = get_common_splits(ds_clue)
# corpus 的训练、验证和测试
corpus_train_ds, corpus_validation_ds, corpus_test_ds = get_common_splits(ds_corpus)

[cluner] 正在从 HuggingFace 下载：xusenlin/clue-ner


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/925 [00:00<?, ?B/s]

data/train-00000-of-00001-39fc4e004146ec(…):   0%|          | 0.00/1.35M [00:00<?, ?B/s]

data/validation-00000-of-00001-bc5663be3(…):   0%|          | 0.00/176k [00:00<?, ?B/s]

...

[corpus] 已导出 JSON：/content/data/corpus/train.json


Creating json from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

[corpus] 已导出 JSON：/content/data/corpus/validation.json


Creating json from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

[corpus] 已导出 JSON：/content/data/corpus/test.json
[corpus] 下载完成并保存至：/content/data/corpus，分片：['train', 'validation', 'test']


In [5]:
def load_local_dataset(subdir: str, base_dir: Path = DATA_DIR) -> DatasetDict:
    """从本地 data/<subdir> 直接加载已下载的数据集（Arrow 格式）。

    用于下载完成后在其他地方快速调取数据，无需联网。

    Args:
        subdir: data 下的子文件夹名，如 "cluner"、"corpus"。
        base_dir: 数据根目录，默认为 DATA_DIR。

    Returns:
        加载后的 DatasetDict。
    """
    target_dir = base_dir / subdir
    if not target_dir.exists():
        raise FileNotFoundError(f"未找到本地数据集：{target_dir}，请先调用 download_dataset 下载。")
    return load_from_disk(str(target_dir))


def load_split_json(subdir: str, split: str = "train", base_dir: Path = DATA_DIR):
    """从本地 JSON 文件加载单个分片，方便快速调取。

    Args:
        subdir: data 下的子文件夹名，如 "cluner"、"corpus"。
        split: 分片名，可选 "train" / "validation" / "test"。
        base_dir: 数据根目录，默认为 DATA_DIR。

    Returns:
        加载后的 Dataset（单个分片）。
    """
    json_path = base_dir / subdir / f"{split}.json"
    if not json_path.exists():
        raise FileNotFoundError(f"未找到 JSON 文件：{json_path}，请先调用 download_dataset 下载。")
    # lines=True 对应导出时的 JSONL 格式
    return load_dataset("json", data_files=str(json_path), split="train")


# 用法示例：
# clue = load_local_dataset("cluner")          # 加载整个 DatasetDict
# clue_train = load_split_json("cluner", "train")  # 只加载训练集 JSON
corpus = load_local_dataset("corpus")

* NER命名实体识别（Named Entity Recognition，NER）是自然语言处理中的一个重要任务，旨在从文本中识别出具有特定意义的实体，如人名、地名、组织机构等。NER的目标是将文本中的实体进行分类和标注，以便后续的分析和处理。
数据集分为text和实体信息，上述流程主要是对文本进行预处理和分析，重点是对实体类型的数据进行分析

# 数据集 BIO标注样式转换 BERT字词对齐

In [6]:
from typing import Optional
import json
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer

# 人民日报（corpus）的标签体系：只有 PER/ORG/LOC 三类，共 7 个标签。
# 数据集里 ner_tags 已经是整数 id，编码顺序与下面 CORPUS_LABELS 完全一致：
#   0=O 1=B-PER 2=I-PER 3=B-ORG 4=I-ORG 5=B-LOC 6=I-LOC
CORPUS_LABELS = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]


def build_corpus_label_schema() -> tuple[list[str], dict[str, int], dict[int, str]]:
    """构建人民日报 NER 的 BIO 标签体系，返回 (labels, label2id, id2label)。"""
    label2id = {lbl: i for i, lbl in enumerate(CORPUS_LABELS)}
    id2label = {i: lbl for lbl, i in label2id.items()}
    return CORPUS_LABELS, label2id, id2label

In [7]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer


class CorpusNerDataset(Dataset):
    """人民日报 NER 数据集（tokens + ner_tags 的逐字 BIO 格式）。

    数据每条形如：
        {"tokens": ["海","钓",...,"厦","门",...],
         "ner_tags": [0, 0, ..., 5, 6, ...]}   # 整数 id，已与 tokens 一一对应

    因为标签已经是逐字 BIO，所以不需要 span_to_bio 转换；
    但仍需做【子词对齐】，把字级标签对齐到 BERT 切出来的 token 上。
    """

    def __init__(
        self,
        records: list,
        tokenizer: BertTokenizer,
        max_length: int = 128,
    ):
        self.records = records
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx: int) -> dict:
        row = self.records[idx]
        tokens = row["tokens"]          # 字列表
        char_labels = row["ner_tags"]   # 字级标签 id，与 tokens 一一对应，直接用

        # 1. 分词：is_split_into_words=True 表示输入已经按字拆好，
        #    让 word_ids() 能把每个 token 映射回它属于第几个字。
        encoding = self.tokenizer(
            tokens,
            is_split_into_words=True,
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        # 2. 子词对齐
        #    word_ids() 形如 [None, 0, 1, 2, 2, 3, ..., None, None]
        #      - None  → [CLS]/[SEP]/[PAD]，没有标签
        #      - 同一个数字连续出现 → 一个字被切成了多个子词（如英文 WTO→W ##T ##O）
        #    规则：每个字只有“第一个子词”用真实标签，特殊 token 和后续子词都标 -100
        #         （-100 是 CrossEntropyLoss 的 ignore_index，不参与 loss 计算）
        word_ids = encoding.word_ids(batch_index=0)
        aligned_labels = []
        prev_word_id = None
        for wid in word_ids:
            if wid is None:
                # 特殊 token：[CLS]/[SEP]/[PAD]
                aligned_labels.append(-100)
            elif wid != prev_word_id:
                # 该字的第一个子词：使用真实标签
                aligned_labels.append(char_labels[wid])
            else:
                # 同一个字的后续子词：忽略
                aligned_labels.append(-100)
            prev_word_id = wid

        # 3. 打包。注意 return_tensors="pt" 后已是 [1, max_length]，
        #    用 squeeze(0) 压成一维 [max_length]，DataLoader 再自动堆成 batch。
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "token_type_ids": encoding["token_type_ids"].squeeze(0),
            "labels": torch.tensor(aligned_labels, dtype=torch.long),
        }


def build_dataloaders(
    tokenizer: BertTokenizer,
    batch_size: int = 32,
    max_length: int = 128,
    train_limit: int = 0,
    eval_limit: int = 0,
) -> tuple[DataLoader, DataLoader, DataLoader]:
    """构建训练/验证/测试 DataLoader，返回 (train_loader, val_loader, test_loader)。

    数据从本地 data/corpus/*.json 读取（需先运行下载 cell）。

    CPU 提速开关（>0 时只取前 N 条，0 表示全量）：
      - train_limit：训练集采样条数
      - eval_limit ：验证/测试集采样条数
    """
    train_records = load_split_json("corpus", "train")
    val_records = load_split_json("corpus", "validation")
    test_records = load_split_json("corpus", "test")

    # 采样：HF Dataset 支持 select，按前 N 条切片
    if train_limit and train_limit > 0:
        train_records = train_records.select(range(min(train_limit, len(train_records))))
    if eval_limit and eval_limit > 0:
        val_records = val_records.select(range(min(eval_limit, len(val_records))))
        test_records = test_records.select(range(min(eval_limit, len(test_records))))

    train_ds = CorpusNerDataset(train_records, tokenizer, max_length)
    val_ds = CorpusNerDataset(val_records, tokenizer, max_length)
    test_ds = CorpusNerDataset(test_records, tokenizer, max_length)

    print(f"数据集规模：训练={len(train_ds)}，验证={len(val_ds)}，测试={len(test_ds)}")

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=0)

    return train_loader, val_loader, test_loader


# 模型
1. BertNER: BERT + 线性分类头, 逐token独立预测BIO标签
  - 过程: input_ids -> Bert -> last_hidden_state -> Dropout -> Linear -> BIO标签
  前向传播过程， 输入文本经过BERT编码器生成上下文相关的表示，然后通过Dropout层进行正则化，最后通过线性分类头预测每个token的BIO标签。
  - 优点: 利用BERT强大的上下文理解能力，适合处理复杂的语言现象。
  - 缺点: 逐token独立预测可能忽略实体之间的依赖关系，可能导致标签不一致。
  - **注意**: 在这里讲一下loss计算的原理, 以免自己遗忘, cross_entropy 函数需要的形参变量主要是输入矩阵和输出类别标签(因为这个是做多类别最常用的函数), input =[N, C]即样本和类别, target是[N]样本的真实类别标签, 计算过程是对每个样本的输入矩阵进行softmax计算得到每个类别的概率分布，拿着labels矩阵为每个样本提供的准确类别索引,代入矩阵, 计算交叉熵损失。
2. BertCRF: BERT + 条件随机场（CRF）层
  - 过程: input_ids -> Bert -> last_hidden_state -> Dropout -> Linear 得到的是发射矩阵 -> CRF -> BIO标签
  前向传播过程，输入文本经过BERT编码器生成上下文相关的表示，然后通过Dropout层进行正则化，接着通过线性层生成特征表示，最后通过CRF层进行全局优化，预测BIO标签序列。
  - **注意**: 损失计算过程中与之前一样是拿经过分类之后的预测标签和crf层的实际标签进行计算, crf记录了真实标签的索引位置, 通过带入矩阵进行计算, 在crf层内部会有一个动态规划算法来计算所有可能的标签序列的概率分布, 以便计算交叉熵损失。
  - 优点: CRF层能够捕捉标签之间的依赖关系，确保预测的标签序列合法。
  - 缺点: 增加了模型的复杂度和训练时间。

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from transformers import BertModel
from pathlib import Path


def _load_bert(bert_path: str) -> BertModel:
    prev = transformers.logging.get_verbosity()
    transformers.logging.set_verbosity_error()
    bert = BertModel.from_pretrained(bert_path)
    transformers.logging.set_verbosity(prev)
    return bert

class BertNER(nn.Module):
    """BERT + 线性分类头，逐 token 独立预测 BIO 标签。

    前向过程：
      input_ids → BertModel → last_hidden_state (B, L, 768)
               → Dropout → Linear(768, num_labels) → logits (B, L, num_labels)

    损失：CrossEntropy，ignore_index=-100 跳过特殊token和非首子词
    预测：argmax(logits, dim=-1)
    """
    def __init__(self, bert_path: str, num_labels: int, dropout: float = 0.1):
        super().__init__()
        self.bert = _load_bert(bert_path)
        hidden_size = self.bert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)
        self.num_labels = num_labels

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        labels: torch.Tensor = None,
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            return_dict=True,
        )
        # 利用bert前向预测出seq的维度
        seq_output = outputs.last_hidden_state  # (B, L, H)
        # 经过线性头分类，在这之前需要经过一层池化进行数据的筛选
        logits = self.classifier(self.dropout(seq_output))  # (B, L, num_labels)

        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits.view(-1, self.num_labels), # [N, C]
                labels.view(-1),                  # [C]
                ignore_index=-100,
            )
        return logits, loss

class BertCRFNER(nn.Module):
    """BERT + CRF 层，全局最优序列解码。

    与 BertNER 的区别：
      - Linear 输出称为 emissions（发射分数），不直接 argmax
      - CRF 在 emissions 上叠加转移矩阵，用 Viterbi 找全局最优序列
      - 损失：负对数似然（CRF 内部计算前向-后向）
      - 解码：self.crf.decode() 返回保证合法的标签序列

    CRF 的约束（自动学习）：
      - 初始只能以 O 或 B-X 开头
      - B-X 之后只能是 I-X 或 B-Y 或 O
      - I-X 之后只能是 I-X 或 B-Y 或 O
    """

    def __init__(self, bert_path: str, num_labels: int, dropout: float = 0.1):
        super().__init__()
        from torchcrf import CRF

        self.bert = _load_bert(bert_path)
        hidden_size = self.bert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)
        self.crf = CRF(num_labels, batch_first=True)
        self.num_labels = num_labels

    def _get_emissions(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
    ) -> torch.Tensor:
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            return_dict=True,
        )
        seq_output = outputs.last_hidden_state
        return self.classifier(self.dropout(seq_output))  # (B, L, num_labels)

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
        labels: torch.Tensor = None,
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        emissions = self._get_emissions(input_ids, attention_mask, token_type_ids)
        mask = attention_mask.bool()  # CRF 要求 BoolTensor

        loss = None
        if labels is not None:
            # CRF 不支持 ignore_index，将 -100 替换为 0（PAD 位置被 mask 屏蔽，不影响梯度）
            labels_crf = labels.clone()
            labels_crf[labels_crf == -100] = 0
            # crf() 返回对数似然（正值），取负得到损失
            loss = -self.crf(emissions, labels_crf, mask=mask, reduction="mean")

        return emissions, loss

    def decode(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        token_type_ids: torch.Tensor,
    ) -> list[list[int]]:
        """Viterbi 解码，返回 list[list[int]]，每条序列长度等于实际token数（不含PAD）。"""
        emissions = self._get_emissions(input_ids, attention_mask, token_type_ids)
        mask = attention_mask.bool()
        return self.crf.decode(emissions, mask=mask)

def build_model(
    use_crf: bool,
    bert_path: str,
    num_labels: int,
    dropout: float = 0.1,
) -> nn.Module:
    """模型工厂函数。"""
    model_cls = BertCRFNER if use_crf else BertNER
    model = model_cls(bert_path=bert_path, num_labels=num_labels, dropout=dropout)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    model_name = "BERT + CRF" if use_crf else "BERT + Linear"
    print(f"模型：{model_name}")
    print(f"  标签数：{num_labels}")
    print(f"  参数总量：{total_params / 1e6:.1f}M")
    print(f"  可训练参数：{trainable_params / 1e6:.1f}M")
    return model

In [10]:
# 训练
import torch
import torch.nn as nn
from torch.optim import AdamW
from transformers import BertTokenizer, get_linear_schedule_with_warmup
from tqdm import tqdm

In [9]:
# ============ 挂载 Google Drive（让训练权重永久保存，不随 Colab 回收而丢失） ============
# 运行后会弹出授权链接，点开用你的 Google 账号授权即可。
# 挂载成功后，下面 SAVE_ROOT 指向你网盘里的 ner_outputs 文件夹。
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    # 你的网盘根目录是 /content/drive/MyDrive，这里在它下面建一个 ner_outputs 文件夹
    SAVE_ROOT = Path('/content/drive/MyDrive/ner_outputs')
    SAVE_ROOT.mkdir(parents=True, exist_ok=True)
    print(f'已挂载 Google Drive，权重将保存到：{SAVE_ROOT}')
except Exception as e:
    # 不在 Colab（比如本地跑）时退回当前目录，避免报错
    SAVE_ROOT = Path.cwd() / 'outputs'
    SAVE_ROOT.mkdir(parents=True, exist_ok=True)
    print(f'未挂载 Drive（{e}），改用本地目录：{SAVE_ROOT}')


Mounted at /content/drive
已挂载 Google Drive，权重将保存到：/content/drive/MyDrive/ner_outputs


# 训练

In [ ]:
import json
import time
from pathlib import Path

import torch
import torch.nn as nn
from torch.optim import AdamW
from transformers import BertTokenizer, get_linear_schedule_with_warmup
from tqdm.auto import tqdm


def evaluate_epoch(
    model: nn.Module,
    loader,
    id2label: dict,
    device: torch.device,
    use_crf: bool,
) -> tuple[float, float]:
    """在 loader 上评估，返回 (avg_loss, entity_f1)。"""
    from seqeval.metrics import f1_score as seqeval_f1

    model.eval()
    total_loss = 0.0
    all_preds: list[list[str]] = []
    all_golds: list[list[str]] = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch["token_type_ids"].to(device)
            labels = batch["labels"].to(device)

            if use_crf:
                emissions, loss = model(input_ids, attention_mask, token_type_ids, labels)
                pred_ids_list = model.decode(input_ids, attention_mask, token_type_ids)
            else:
                logits, loss = model(input_ids, attention_mask, token_type_ids, labels)
                pred_ids_list = logits.argmax(dim=-1).tolist()

            total_loss += loss.item()

            # CRF decode 用 attention_mask 解码，pred_ids 长度 = 非 PAD token 数；
            # 在前 len(pred_ids) 个位置上，pred_ids 与 labels 是一一对应的（含 CLS/SEP）。
            # 非 CRF 时 pred_ids 与 labels 同 max_length。
            # 两种情况都直接按下标 j 取预测，再用 gold==-100 过滤掉 CLS/SEP/非首子词/PAD。
            labels_np = labels.cpu().tolist()
            for i in range(len(input_ids)):
                gold_seq, pred_seq = [], []
                token_labels = labels_np[i]
                pred_ids = pred_ids_list[i]
                for j, gold_id in enumerate(token_labels):
                    if gold_id == -100:
                        continue
                    if j >= len(pred_ids):
                        break
                    gold_seq.append(id2label[gold_id])
                    pred_seq.append(id2label.get(pred_ids[j], "O"))
                all_golds.append(gold_seq)
                all_preds.append(pred_seq)

    avg_loss = total_loss / len(loader)
    entity_f1 = seqeval_f1(all_golds, all_preds)
    return avg_loss, entity_f1


def train_one_epoch(
    model: nn.Module,
    loader,
    optimizer,
    scheduler,
    device: torch.device,
    epoch: int,
    total_epochs: int,
    grad_accum: int,
) -> float:
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(loader, desc=f"Epoch {epoch}/{total_epochs} [Train]", leave=False)
    for step, batch in enumerate(pbar):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        labels = batch["labels"].to(device)

        _, loss = model(input_ids, attention_mask, token_type_ids, labels)

        (loss / grad_accum).backward()
        total_loss += loss.item()

        if (step + 1) % grad_accum == 0:
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    # 处理最后不足 grad_accum 的批次
    if len(loader) % grad_accum != 0:
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    return total_loss / len(loader)


# ============ 训练配置（notebook 直接改这里，无需命令行） ============
USE_CRF = True                # True 用 BERT+CRF，False 用 BERT+Linear

# 【是否有 GPU】CPU 环境会自动切到“轻量配置”，让训练能在十几分钟内跑完。
HAS_GPU = torch.cuda.is_available()

if HAS_GPU:
    # —— GPU：用完整的 bert-base-chinese + 全量数据 ——
    BERT_NAME = "bert-base-chinese"
    EPOCHS = 3
    BATCH_SIZE = 32
    MAX_LENGTH = 128
    TRAIN_LIMIT = 0        # 0 = 全量 20865 条
    EVAL_LIMIT = 0         # 0 = 全量验证集
else:
    # —— CPU：换 3 层小模型 + 短句长 + 数据采样 + 单轮，CPU 也能盯着跑完 ——
    BERT_NAME = "hfl/rbt3"   # 3 层中文 RoBERTa，约 bert-base 的 1/4，快 3~4 倍
    EPOCHS = 1
    BATCH_SIZE = 16
    MAX_LENGTH = 64          # 人民日报句子大多不长，64 足够
    TRAIN_LIMIT = 3000       # 训练集只取前 3000 条 修改成为0即全训练
    EVAL_LIMIT = 500         # 验证/测试各取前 500 条  同理

LR = 2e-5                        # BERT 层学习率
HEAD_LR_MULT = 5.0               # 分类头学习率倍数
WARMUP_RATIO = 0.1
GRAD_ACCUM = 1
DROPOUT = 0.1

# 输出目录：用上一个 cell 挂载好的 SAVE_ROOT（Google Drive，永久保存）。
# 若没运行挂载 cell，则退回当前目录下的 outputs/，避免 NameError。
try:
    OUTPUT_DIR = SAVE_ROOT
except NameError:
    OUTPUT_DIR = Path.cwd() / "outputs"
CKPT_DIR = OUTPUT_DIR / "checkpoints"
LOG_DIR = OUTPUT_DIR / "logs"

device = torch.device("cuda" if HAS_GPU else "cpu")
print(f"设备：{device} | 模型：{BERT_NAME} | epochs={EPOCHS} | "
      f"max_len={MAX_LENGTH} | train_limit={TRAIN_LIMIT or '全量'}")
print(f"权重保存目录：{OUTPUT_DIR}")

# 1. 标签体系（人民日报 7 个标签）
labels, label2id, id2label = build_corpus_label_schema()
num_labels = len(labels)
print(f"BIO 标签数：{num_labels}（{labels}）")

# 2. Tokenizer（从 HF 自动下载）
tokenizer = BertTokenizer.from_pretrained(BERT_NAME)

# 3. DataLoader（CPU 下按 TRAIN_LIMIT/EVAL_LIMIT 采样）
train_loader, val_loader, test_loader = build_dataloaders(
    tokenizer=tokenizer,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
    train_limit=TRAIN_LIMIT,
    eval_limit=EVAL_LIMIT,
)

# 4. 模型
model = build_model(
    use_crf=USE_CRF,
    bert_path=BERT_NAME,
    num_labels=num_labels,
    dropout=DROPOUT,
).to(device)

# 5. 分层学习率：BERT 层用基础 lr，分类头用 HEAD_LR_MULT 倍
bert_params = list(model.bert.parameters())
head_params = (
    list(model.classifier.parameters()) +
    list(model.dropout.parameters()) +
    (list(model.crf.parameters()) if USE_CRF else [])
)
optimizer = AdamW(
    [
        {"params": bert_params, "lr": LR},
        {"params": head_params, "lr": LR * HEAD_LR_MULT},
    ],
    weight_decay=0.01,
)

total_steps = len(train_loader) * EPOCHS // GRAD_ACCUM
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
print(f"\n训练步数：{total_steps}，预热步数：{warmup_steps}")

# 6. 训练循环
run_tag = "crf" if USE_CRF else "linear"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
ckpt_path = CKPT_DIR / f"best_{run_tag}.pt"
log_path = LOG_DIR / f"train_{run_tag}.json"

best_f1 = 0.0
log_records = []

print(f"\n开始训练（{'BERT+CRF' if USE_CRF else 'BERT+Linear'}）...")
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss = train_one_epoch(
        model, train_loader, optimizer, scheduler, device,
        epoch, EPOCHS, GRAD_ACCUM
    )
    val_loss, val_f1 = evaluate_epoch(model, val_loader, id2label, device, USE_CRF)
    elapsed = time.time() - t0

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_entity_f1={val_f1:.4f} | "
        f"time={elapsed:.0f}s"
    )

    log_records.append({
        "epoch": epoch,
        "train_loss": round(train_loss, 6),
        "val_loss": round(val_loss, 6),
        "val_entity_f1": round(val_f1, 6),
        "elapsed_s": round(elapsed, 1),
    })

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(
            {
                "epoch": epoch,
                "use_crf": USE_CRF,
                "bert_name": BERT_NAME,
                "state_dict": model.state_dict(),
                "val_entity_f1": val_f1,
                "label2id": label2id,
                "id2label": id2label,
            },
            ckpt_path,
        )
        print(f"  ★ 新最优 F1={val_f1:.4f}，已保存 → {ckpt_path}")

with open(log_path, "w", encoding="utf-8") as f:
    json.dump(log_records, f, ensure_ascii=False, indent=2)

print(f"\n训练完成！最优 val_entity_f1={best_f1:.4f}")
print(f"  Checkpoint: {ckpt_path}")
print(f"  训练日志:   {log_path}")


# 评估

In [ ]:
import torch
import torch.nn as nn
from tqdm.auto import tqdm


def train_one_epoch(
    model: nn.Module,
    loader,
    optimizer,
    scheduler,
    device: torch.device,
    epoch: int,
    total_epochs: int,
    grad_accum: int,
) -> float:
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(loader, desc=f"Epoch {epoch}/{total_epochs} [Train]", leave=False)
    for step, batch in enumerate(pbar):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        labels         = batch["labels"].to(device)

        _, loss = model(input_ids, attention_mask, token_type_ids, labels)
        (loss / grad_accum).backward()
        total_loss += loss.item()

        if (step + 1) % grad_accum == 0:
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    if len(loader) % grad_accum != 0:
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    return total_loss / len(loader)


def evaluate_epoch(
    model: nn.Module,
    loader,
    id2label: dict,
    device: torch.device,
    use_crf: bool,
) -> tuple[float, float]:
    from seqeval.metrics import f1_score as seqeval_f1

    model.eval()
    total_loss = 0.0
    all_preds: list[list[str]] = []
    all_golds: list[list[str]] = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch["token_type_ids"].to(device)
            labels         = batch["labels"].to(device)

            if use_crf:
                _, loss       = model(input_ids, attention_mask, token_type_ids, labels)
                pred_ids_list = model.decode(input_ids, attention_mask, token_type_ids)
            else:
                logits, loss  = model(input_ids, attention_mask, token_type_ids, labels)
                pred_ids_list = logits.argmax(dim=-1).tolist()

            total_loss += loss.item()

            # CRF decode 用 attention_mask 解码，pred_ids 长度 = 非 PAD token 数；
            # 在前 len(pred_ids) 个位置上，pred_ids 与 labels 是一一对应的（含 CLS/SEP）。
            # 非 CRF 时 pred_ids 与 labels 同 max_length。
            # 两种情况都直接按下标 j 取预测，再用 gold==-100 过滤掉 CLS/SEP/非首子词/PAD。
            for i, token_labels in enumerate(labels.cpu().tolist()):
                gold_seq, pred_seq = [], []
                pred_ids = pred_ids_list[i]
                for j, gold_id in enumerate(token_labels):
                    if gold_id == -100:
                        continue
                    if j >= len(pred_ids):
                        break
                    gold_seq.append(id2label[gold_id])
                    pred_seq.append(id2label.get(pred_ids[j], "O"))
                all_golds.append(gold_seq)
                all_preds.append(pred_seq)

    return total_loss / len(loader), seqeval_f1(all_golds, all_preds)
# ── 模型选择 ──────────────────────────────────────────────
USE_CRF   = True
BERT_NAME = "bert-base-chinese"   # CPU 备选: "hfl/rbt3"

# ── 数据量限制（0 = 全量） ────────────────────────────────
TRAIN_LIMIT = 0
EVAL_LIMIT  = 0

# ── 训练超参 ──────────────────────────────────────────────
EPOCHS       = 3
BATCH_SIZE   = 32
MAX_LENGTH   = 128
LR           = 2e-5
HEAD_LR_MULT = 5.0
WARMUP_RATIO = 0.1
GRAD_ACCUM   = 1
DROPOUT      = 0.1

from pathlib import Path

# 选项 A：已挂载 Google Drive
CKPT_DIR = Path("/content/drive/MyDrive/ner_outputs/checkpoints")
LOG_DIR = Path("/content/drive/MyDrive/ner_outputs/logs")
# 选项 B：Colab 本地临时目录（重启后丢失，但不依赖 Drive）
# CKPT_DIR = Path("outputs/checkpoints")
# LOG_DIR  = Path("outputs/logs")

CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

run_tag   = "crf" if USE_CRF else "linear"
ckpt_path = CKPT_DIR / f"best_{run_tag}.pt"
log_path  = LOG_DIR  / f"train_{run_tag}.json"

print(f"Checkpoint → {ckpt_path}")
print(f"日志       → {log_path}")


In [ ]:
# ============ 在验证/测试集上评估（entity-level F1 + 非法序列统计） ============
# 直接读上一步保存到 Drive 的 checkpoint，无需命令行参数。
import json
from collections import defaultdict

import torch
from transformers import BertTokenizer
from seqeval.metrics import (
    f1_score, precision_score, recall_score,
    classification_report as seqeval_report,
)


def count_illegal_sequences(pred_seqs: list[list[str]]) -> dict:
    """统计非法 BIO 序列数量。

    非法类型：
      - illegal_start：序列以 I-X 开头
      - illegal_transition：O→I-X，或 B-X/I-X 后跟 I-Y（X≠Y）
    线性头通常有一些非法序列，CRF 充分训练后为 0。
    """
    stats = {"illegal_start": 0, "illegal_transition": 0, "total_seqs": len(pred_seqs)}
    for seq in pred_seqs:
        if not seq:
            continue
        if seq[0].startswith("I-"):
            stats["illegal_start"] += 1
        for i in range(1, len(seq)):
            prev, curr = seq[i - 1], seq[i]
            if curr.startswith("I-"):
                curr_type = curr[2:]
                if prev == "O":
                    stats["illegal_transition"] += 1
                elif prev.startswith("B-") or prev.startswith("I-"):
                    if prev[2:] != curr_type:
                        stats["illegal_transition"] += 1
    stats["total_illegal"] = stats["illegal_start"] + stats["illegal_transition"]
    return stats


def run_inference(model, loader, id2label, device, use_crf):
    """在 loader 上推理，返回 (all_preds, all_golds)，元素为字符串标签列表。"""
    model.eval()
    all_preds, all_golds = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch["token_type_ids"].to(device)
            labels = batch["labels"].to(device)

            if use_crf:
                pred_ids_list = model.decode(input_ids, attention_mask, token_type_ids)
            else:
                logits, _ = model(input_ids, attention_mask, token_type_ids)
                pred_ids_list = logits.argmax(dim=-1).tolist()

            labels_list = labels.cpu().tolist()
            for i in range(len(input_ids)):
                gold_seq, pred_seq = [], []
                token_labels = labels_list[i]
                pred_ids = pred_ids_list[i]

                # CRF decode 用 attention_mask 解码，pred_ids 长度 = 非 PAD token 数；
                # 在前 len(pred_ids) 个位置上，pred_ids 与 labels 是一一对应的（含 CLS/SEP）。
                # 非 CRF 时 pred_ids 与 labels 同 max_length。
                # 两种情况都直接按下标 j 取预测，再用 gold==-100 过滤掉 CLS/SEP/非首子词/PAD。
                for j, gold_id in enumerate(token_labels):
                    if gold_id == -100:
                        continue
                    if j >= len(pred_ids):
                        break
                    gold_seq.append(id2label[gold_id])
                    pred_seq.append(id2label.get(pred_ids[j], "O"))

                all_golds.append(gold_seq)
                all_preds.append(pred_seq)

    return all_preds, all_golds


# ---------- 评估配置 ----------
EVAL_USE_CRF = True         # 复用训练时的设置（与 checkpoint 对应）
EVAL_SPLIT = "validation"       # "validation" 或 "test"

run_tag = "crf" if EVAL_USE_CRF else "linear"
ckpt_path = CKPT_DIR / f"best_{run_tag}.pt"

if not ckpt_path.exists():
    print(f"找不到 checkpoint：{ckpt_path}\n请先运行训练 cell（USE_CRF={EVAL_USE_CRF}）。")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)

    # 标签体系（人民日报 7 类）
    labels, label2id, id2label = build_corpus_label_schema()

    # 用 checkpoint 里记录的 bert_name（没有则退回当前 BERT_NAME）
    eval_bert_name = ckpt.get("bert_name", BERT_NAME)

    model = build_model(
        use_crf=EVAL_USE_CRF,
        bert_path=eval_bert_name,
        num_labels=len(labels),
    ).to(device)
    model.load_state_dict(ckpt["state_dict"])
    print(f"已加载 checkpoint（epoch={ckpt['epoch']}，val_f1={ckpt['val_entity_f1']:.4f}，模型={eval_bert_name}）")

    # 构建 DataLoader（评估同样用采样开关，保持与训练一致的规模）
    tokenizer = BertTokenizer.from_pretrained(eval_bert_name)
    _, val_loader, test_loader = build_dataloaders(
        tokenizer=tokenizer,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        eval_limit=EVAL_LIMIT,
    )
    loader = val_loader if EVAL_SPLIT == "validation" else test_loader

    print(f"\n正在在 [{EVAL_SPLIT}] 集上推理...")
    all_preds, all_golds = run_inference(model, loader, id2label, device, EVAL_USE_CRF)

    # seqeval entity-level 指标
    p = precision_score(all_golds, all_preds)
    r = recall_score(all_golds, all_preds)
    f1 = f1_score(all_golds, all_preds)

    print("\n" + "=" * 70)
    print(f"模型：{'BERT + CRF' if EVAL_USE_CRF else 'BERT + Linear'}  |  评估集：{EVAL_SPLIT}")
    print("=" * 70)
    print(f"Entity-level Precision: {p:.4f}")
    print(f"Entity-level Recall:    {r:.4f}")
    print(f"Entity-level F1:        {f1:.4f}")

    print("\n【逐类型 F1】")
    print(seqeval_report(all_golds, all_preds, digits=4))

    # 非法序列统计
    illegal_stats = count_illegal_sequences(all_preds)
    print("【非法 BIO 序列统计】")
    print(f"  总序列数：{illegal_stats['total_seqs']}")
    print(f"  非法开头（I-X 开头）：{illegal_stats['illegal_start']} 条")
    print(f"  非法转移（B-X/I-X → I-Y, X≠Y）：{illegal_stats['illegal_transition']} 条")
    print(f"  合计非法序列：{illegal_stats['total_illegal']} 条")
    pct = illegal_stats["total_illegal"] / max(illegal_stats["total_seqs"], 1) * 100
    if EVAL_USE_CRF:
        if illegal_stats["total_illegal"] == 0:
            print("  → CRF Viterbi 解码：非法序列 0 条 ✓（转移矩阵已学到 BIO 约束）")
        else:
            print(f"  → CRF 非法序列 {illegal_stats['total_illegal']} 条（{pct:.1f}%），多训几轮可降至 0")
    else:
        print(f"  → 线性头约 {pct:.1f}% 的序列含非法转移，充分训练的 CRF 可完全消除")

    # 保存评估结果到 Drive
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    result = {
        "model": "BERT+CRF" if EVAL_USE_CRF else "BERT+Linear",
        "split": EVAL_SPLIT,
        "precision": round(p, 6),
        "recall": round(r, 6),
        "f1": round(f1, 6),
        "illegal_stats": illegal_stats,
    }
    out_path = LOG_DIR / f"eval_{run_tag}_{EVAL_SPLIT}.json"
    with open(out_path, "w", encoding="utf-8") as fout:
        json.dump(result, fout, ensure_ascii=False, indent=2)
    print(f"\n评估结果已保存 → {out_path}")


In [ ]:
# ============ CRF 退化诊断：查清 F1=0.02 的真因 ============
# 核心思想：F1 接近 0 但 illegal_transition 也接近 0，且 P==R，
# 几乎只能是 CRF 退化解（全部预测 O / 全部预测同一标签）。
# 这个 cell 直接打印预测分布、对齐情况、loss 曲线，3 分钟内定位。
import json
from collections import Counter
from pathlib import Path

import torch

assert "model" in globals(), "请先运行评估 cell 加载 model 后再跑诊断"
device = next(model.parameters()).device
model.eval()

labels, label2id, id2label = build_corpus_label_schema()

# ── 1. 取一个真实 batch，对比 CRF 输出与 Linear-style argmax ──────────────────
batch = next(iter(val_loader))
input_ids      = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
token_type_ids = batch["token_type_ids"].to(device)
gold_labels    = batch["labels"]

with torch.no_grad():
    emissions, loss = model(input_ids, attention_mask, token_type_ids,
                            batch["labels"].to(device))
    crf_path  = model.decode(input_ids, attention_mask, token_type_ids)
    arg_path  = emissions.argmax(dim=-1).cpu().tolist()

print("─" * 70)
print(f"loss on this batch        : {loss.item():.4f}")
print(f"emissions.shape           : {tuple(emissions.shape)}  (B, L, num_labels)")
print(f"CRF decode 第 0 条长度    : {len(crf_path[0])}")
print(f"argmax 第 0 条长度        : {len(arg_path[0])}")
print(f"attention_mask[0].sum()   : {attention_mask[0].sum().item()}")
print(f"labels[0] 中非 -100 个数  : {(gold_labels[0] != -100).sum().item()}")

# ── 2. 预测标签分布：是不是全部退化成 O / B-LOC ──────────────────────────────
all_preds_flat = [p for seq in crf_path for p in seq]
all_argmax_flat = []
for i in range(len(arg_path)):
    L = attention_mask[i].sum().item()
    all_argmax_flat.extend(arg_path[i][:L])

pred_dist = Counter(all_preds_flat)
argm_dist = Counter(all_argmax_flat)
gold_flat = gold_labels[gold_labels != -100].tolist()
gold_dist = Counter(gold_flat)

print("\n标签分布（这一个 batch 内）：")
print(f"{'label':<10} {'gold':>6} {'CRF decode':>12} {'argmax(emis)':>14}")
for lid, name in id2label.items():
    print(f"{name:<10} {gold_dist.get(lid,0):>6} "
          f"{pred_dist.get(lid,0):>12} {argm_dist.get(lid,0):>14}")

# 退化判定
top_pred = pred_dist.most_common(1)[0]
ratio = top_pred[1] / max(sum(pred_dist.values()), 1)
print(f"\nCRF 输出占比最高的标签：{id2label[top_pred[0]]} = {ratio*100:.1f}%")
if ratio > 0.95:
    print(f"  ⚠️  退化解：CRF 几乎全输出 [{id2label[top_pred[0]]}]，转移矩阵没学好。")
elif id2label[top_pred[0]] == "O" and ratio > 0.85:
    print("  ⚠️  CRF 极度偏向 O，等价于'什么都不识别'，必然导致 P/R 双双≈0。")

# ── 3. CRF vs argmax 的差异：转移矩阵到底起了什么作用 ────────────────────────
B, L = input_ids.shape
diff = 0
for i in range(B):
    n = attention_mask[i].sum().item()
    for j in range(n):
        if crf_path[i][j] != arg_path[i][j]:
            diff += 1
total = sum(attention_mask[i].sum().item() for i in range(B))
print(f"\nCRF 与 argmax(emis) 不同的 token 数: {diff}/{total} ({diff/total*100:.1f}%)")
if diff / total < 0.01:
    print("  → 转移矩阵几乎没在改写 emissions，CRF 退化为单纯的 argmax。")

# ── 4. 看 3 条样本的实际预测 vs 金标 ─────────────────────────────────────────
print("\n样本对比（前 3 条，只看前 30 个非特殊 token）：")
for i in range(min(3, B)):
    gold_str, pred_str, argm_str = [], [], []
    n = attention_mask[i].sum().item()
    for j in range(n):
        gid = gold_labels[i][j].item()
        if gid == -100:
            continue
        gold_str.append(id2label[gid])
        pred_str.append(id2label[crf_path[i][j]])
        argm_str.append(id2label[arg_path[i][j]])
        if len(gold_str) >= 30:
            break
    print(f"\n  [#{i}]")
    print(f"    GOLD  : {' '.join(gold_str)}")
    print(f"    CRF   : {' '.join(pred_str)}")
    print(f"    ARGMAX: {' '.join(argm_str)}")

# ── 5. 训练日志：loss 是不是根本没降 ─────────────────────────────────────────
log_path = LOG_DIR / "train_crf.json"
if log_path.exists():
    with open(log_path, encoding="utf-8") as f:
        recs = json.load(f)
    print("\n训练日志（train_crf.json）：")
    for r in recs:
        print(f"  epoch {r['epoch']}: train_loss={r['train_loss']:.4f}  "
              f"val_loss={r['val_loss']:.4f}  val_f1={r['val_entity_f1']:.4f}")
    if recs and recs[0]["train_loss"] - recs[-1]["train_loss"] < 0.5:
        print("  ⚠️  loss 几乎没降——CRF 没收敛。"
              "建议：CRF 层 lr 提到 1e-3（远高于 BERT），或多训几轮。")
else:
    print(f"\n（找不到 {log_path}，无法看 loss 曲线）")


In [13]:
# ============ 预测 demo：输入一句话 → 识别人名/地名/机构 ============
# 复用上面已加载的 model / tokenizer / id2label / device（若没运行评估 cell，会自动加载）。
import torch

# 标签英文 → 中文，方便展示
TYPE_CN = {"PER": "人名", "ORG": "机构", "LOC": "地名"}


def _ensure_model_loaded():
    """如果当前内核里还没有训练好的 model，就从 Drive 的 checkpoint 加载一个。"""
    global model, tokenizer, id2label, device
    if "model" in globals() and "tokenizer" in globals():
        return  # 评估 cell 已经准备好了，直接用

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    run_tag = "crf" if USE_CRF else "linear"
    ckpt_path = CKPT_DIR / f"best_{run_tag}.pt"
    if not ckpt_path.exists():
        raise FileNotFoundError(f"找不到 checkpoint：{ckpt_path}，请先运行训练 cell。")

    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    _, _, id2label = build_corpus_label_schema()
    eval_bert_name = ckpt.get("bert_name", BERT_NAME)
    from transformers import BertTokenizer
    tokenizer = BertTokenizer.from_pretrained(eval_bert_name)
    model = build_model(use_crf=USE_CRF, bert_path=eval_bert_name,
                        num_labels=len(id2label)).to(device)
    model.load_state_dict(ckpt["state_dict"])


def predict(text: str) -> list[dict]:
    """对一句话做 NER，返回实体列表 [{'entity':..,'type':..,'start':..,'end':..}]。"""
    _ensure_model_loaded()
    model.eval()

    # 按字拆分（与训练时一致：人民日报是逐字标注）
    chars = list(text)
    encoding = tokenizer(
        chars,
        is_split_into_words=True,
        max_length=MAX_LENGTH,
        truncation=True,
        return_tensors="pt",
    )
    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)
    token_type_ids = encoding["token_type_ids"].to(device)

    with torch.no_grad():
        if USE_CRF:
            pred_ids = model.decode(input_ids, attention_mask, token_type_ids)[0]
        else:
            logits, _ = model(input_ids, attention_mask, token_type_ids)
            pred_ids = logits.argmax(dim=-1)[0].tolist()

    # 把 token 级预测对齐回“每个字”的标签（只取每个字的第一个子词）
    word_ids = encoding.word_ids(batch_index=0)
    char_tags = [None] * len(chars)
    prev = None
    for pos, wid in enumerate(word_ids):
        if wid is None or wid == prev:
            prev = wid
            continue
        if wid < len(chars):
            # CRF 的 pred_ids 已去掉 PAD 但仍含 CLS/SEP，索引与 word_ids 的 pos 对齐
            idx = pos if pos < len(pred_ids) else len(pred_ids) - 1
            char_tags[wid] = id2label.get(pred_ids[idx], "O")
        prev = wid

    # BIO 解码：把连续的 B-/I- 合并成实体
    entities = []
    cur = None  # {'type','start','chars'}
    for i, tag in enumerate(char_tags):
        tag = tag or "O"
        if tag.startswith("B-"):
            if cur:
                entities.append(cur)
            cur = {"type": tag[2:], "start": i, "chars": [chars[i]]}
        elif tag.startswith("I-") and cur and cur["type"] == tag[2:]:
            cur["chars"].append(chars[i])
        else:
            if cur:
                entities.append(cur)
            cur = None
    if cur:
        entities.append(cur)

    return [
        {"entity": "".join(e["chars"]), "type": e["type"],
         "start": e["start"], "end": e["start"] + len(e["chars"])}
        for e in entities
    ]


def show(text: str):
    """打印识别结果，并把实体在原文里高亮（用【类型|实体】标出）。"""
    ents = predict(text)
    # 1. 标注版原文
    marked, last = "", 0
    for e in sorted(ents, key=lambda x: x["start"]):
        marked += text[last:e["start"]]
        cn = TYPE_CN.get(e["type"], e["type"])
        marked += f"【{cn}|{e['entity']}】"
        last = e["end"]
    marked += text[last:]
    print("原文：", text)
    print("标注：", marked)
    # 2. 实体清单
    if ents:
        print("实体：")
        for e in ents:
            cn = TYPE_CN.get(e["type"], e["type"])
            print(f"   - {e['entity']}  （{cn} {e['type']}，位置 {e['start']}~{e['end']}）")
    else:
        print("实体： （未识别到实体）")
    print("-" * 50)


# ---------- 试几句 ----------
show("李雷和韩梅梅在北京大学的图书馆里讨论问题。")
show("中国科学院计算技术研究所位于北京市海淀区。")
show("马云在杭州创办了阿里巴巴。")

# 你也可以自己输入：
# show("这里换成你想测试的句子")


原文： 李雷和韩梅梅在北京大学的图书馆里讨论问题。
标注： 【人名|李雷】和【人名|韩梅梅】在【机构|北京大学】的图书馆里讨论问题。
实体：
   - 李雷  （人名 PER，位置 0~2）
   - 韩梅梅  （人名 PER，位置 3~6）
   - 北京大学  （机构 ORG，位置 7~11）
--------------------------------------------------
原文： 中国科学院计算技术研究所位于北京市海淀区。
标注： 【机构|中国科学院计算技术研究所】位于【地名|北京市】【地名|海淀区】。
实体：
   - 中国科学院计算技术研究所  （机构 ORG，位置 0~12）
   - 北京市  （地名 LOC，位置 14~17）
   - 海淀区  （地名 LOC，位置 17~20）
--------------------------------------------------
原文： 马云在杭州创办了阿里巴巴。
标注： 【人名|马云】在【地名|杭州】创办了【机构|阿里巴巴】。
实体：
   - 马云  （人名 PER，位置 0~2）
   - 杭州  （地名 LOC，位置 3~5）
   - 阿里巴巴  （机构 ORG，位置 8~12）
--------------------------------------------------


# 生成式 NER：LoRA SFT 微调 Qwen2-0.5B-Instruct（人民日报 corpus）

与上面的 **BERT 序列标注** 是两条不同的技术路线，做同一份人民日报 NER 数据：

| | BERT + Linear/CRF（上面） | LLM SFT（这里） |
|---|---|---|
| 范式 | 序列标注，逐字打 BIO 标签 | 生成式，直接「写出」实体 JSON |
| 输出 | 每个字一个标签 id | `{"entities": [{"text":"北京","type":"LOC"}]}` |
| Loss | 字级 CrossEntropy | 只在 JSON 输出部分算 loss（prompt 全 -100） |
| 微调 | 全参数 | LoRA（仅 ~0.2% 参数）/ 可选全量 |

**数据转换**：人民日报是 `tokens` + `ner_tags`（整数 BIO），这里先把 BIO 解码成实体列表，再拼成目标 JSON 字符串作为 SFT 的 TARGET。实体类型只有 3 类：PER（人名）、ORG（机构）、LOC（地名）。

In [ ]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 36.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
# ============ LLM SFT（LoRA 微调）——人民日报 corpus 版 ============
# 教学重点：
#   1. NER 的指令微调格式：输入文本 → 输出实体 JSON 列表（不是单个类别名）
#   2. Loss masking：prompt 全 -100，只在 JSON 输出部分算 loss
#   3. LoRA 高效微调：可训练参数 ~0.2%（FULL_FT=True 走全量）
#   4. 预 tokenize：训练前把所有数据一次性转成 ID，__getitem__ 只做列表切片
import json
import random
import time
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm

try:
    from peft import get_peft_model, LoraConfig, TaskType
    PEFT_AVAILABLE = True
except ImportError:
    PEFT_AVAILABLE = False


# 人民日报只有 3 类实体；ner_tags 整数编码同 build_corpus_label_schema：
#   0=O 1=B-PER 2=I-PER 3=B-ORG 4=I-ORG 5=B-LOC 6=I-LOC
SFT_TYPE_CN = {"PER": "人名", "ORG": "机构", "LOC": "地名"}

SYSTEM_PROMPT = (
    "你是一个命名实体识别助手。从文本中识别命名实体，以 JSON 格式输出。\n"
    "实体类型（英文标识）：PER（人名）、ORG（机构）、LOC（地名）\n"
    '输出格式（严格遵守，不输出其他内容）：{"entities": [{"text": "实体文本", "type": "实体类型"}]}\n'
    '无实体时输出：{"entities": []}'
)


def bio_to_entities(tokens: list, tag_ids: list, id2label: dict) -> list:
    """把逐字 BIO（整数 id 列表）解码成实体列表 [{'text','type'}]。"""
    entities = []
    cur_type, cur_chars = None, []
    for tok, tid in zip(tokens, tag_ids):
        tag = id2label.get(tid, "O")
        if tag.startswith("B-"):
            if cur_type:
                entities.append({"text": "".join(cur_chars), "type": cur_type})
            cur_type, cur_chars = tag[2:], [tok]
        elif tag.startswith("I-") and cur_type == tag[2:]:
            cur_chars.append(tok)
        else:
            if cur_type:
                entities.append({"text": "".join(cur_chars), "type": cur_type})
            cur_type, cur_chars = None, []
    if cur_type:
        entities.append({"text": "".join(cur_chars), "type": cur_type})
    return entities


def record_to_text_and_target(record: dict, id2label: dict) -> tuple[str, str]:
    """corpus 单条 → (原文文本, 目标 JSON 字符串)。"""
    tokens = record["tokens"]
    text = "".join(tokens)
    entities = bio_to_entities(tokens, record["ner_tags"], id2label)
    target = json.dumps({"entities": entities}, ensure_ascii=False)
    return text, target


# ══════════════════════════════════════════════════════════════════════════════
# 预 tokenize：训练前一次性把所有记录转成 ID，缓存在内存里
# ══════════════════════════════════════════════════════════════════════════════

def pretokenize_records(records, tokenizer, id2label) -> list:
    """把 corpus 记录批量转换为 (prompt_ids, response_ids) 元组列表。

    在 DataLoader 启动前一次性完成所有 tokenize，__getitem__ 只做数组切片，
    彻底避免训练时在每个 step 里重复调用 tokenizer，显著提升数据吞吐速度。

    返回：[(prompt_ids: list[int], response_ids: list[int]), ...]
    """
    items = []
    for record in tqdm(records, desc="预 tokenize", leave=False):
        text, target = record_to_text_and_target(record, id2label)
        prompt_text = tokenizer.apply_chat_template(
            [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": text},
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
        response_ids = (
            tokenizer.encode(target, add_special_tokens=False)
            + [tokenizer.eos_token_id]
        )
        items.append((prompt_ids, response_ids))
    return items


# ══════════════════════════════════════════════════════════════════════════════
# Dataset：接收预 tokenize 好的列表，__getitem__ 只做截断 + 标签拼接
# ══════════════════════════════════════════════════════════════════════════════

class SFTDataset(Dataset):
    """接收预先 tokenize 好的 (prompt_ids, response_ids) 列表。

    __getitem__ 只做截断 + 标签拼接（纯 Python 列表切片），不持有 tokenizer，
    DataLoader worker 进程无需重复序列化/持有 tokenizer 对象。

    Loss mask 结构：
      prompt_ids          → -100（prompt，不算 loss）
      response_ids + EOS  → 真实 id（只在这里算 loss）
    """

    def __init__(self, tokenized_items: list, max_length: int = 256):
        self.items = tokenized_items
        self.max_length = max_length

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        prompt_ids, response_ids = self.items[idx]
        input_ids = (prompt_ids + response_ids)[: self.max_length]
        labels    = ([-100] * len(prompt_ids) + response_ids)[: self.max_length]
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels":    torch.tensor(labels,    dtype=torch.long),
        }


def collate_fn(batch, pad_id):
    """右侧 padding 到 batch 内最长，labels 用 -100 填充。"""
    max_len = max(item["input_ids"].size(0) for item in batch)
    input_ids_list, labels_list, mask_list = [], [], []
    for item in batch:
        n = item["input_ids"].size(0)
        pad = max_len - n
        input_ids_list.append(torch.cat([item["input_ids"],
                                          torch.full((pad,), pad_id, dtype=torch.long)]))
        labels_list.append(torch.cat([item["labels"],
                                       torch.full((pad,), -100, dtype=torch.long)]))
        mask_list.append(torch.cat([torch.ones(n, dtype=torch.long),
                                     torch.zeros(pad, dtype=torch.long)]))
    return {
        "input_ids":      torch.stack(input_ids_list),
        "labels":         torch.stack(labels_list),
        "attention_mask": torch.stack(mask_list),
    }


# ══════════════════════════════════════════════════════════════════════════════
# 配置（notebook 直接改这里，无需命令行）
# ══════════════════════════════════════════════════════════════════════════════
SFT_MODEL_NAME = "Qwen/Qwen2-0.5B-Instruct"
FULL_FT = False                 # True=全量微调（需显存≥16G）；False=LoRA
SEED = 42

HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    SFT_EPOCHS = 3
    SFT_BATCH_SIZE = 4
    SFT_MAX_LENGTH = 256
    SFT_TRAIN_LIMIT = 0          # 0 = 全量 20865 条
    SFT_EVAL_LIMIT = 300
else:
    # CPU：0.5B LLM 很慢，小规模演示
    SFT_EPOCHS = 1
    SFT_BATCH_SIZE = 2
    SFT_MAX_LENGTH = 192
    SFT_TRAIN_LIMIT = 500
    SFT_EVAL_LIMIT = 100

GRAD_ACCUM = 4
SFT_LR = 2e-5 if FULL_FT else 2e-4
LORA_R, LORA_ALPHA = 8, 16

try:
    SFT_OUTPUT_DIR = SAVE_ROOT
except NameError:
    SFT_OUTPUT_DIR = Path.cwd() / "outputs"
SFT_CKPT_DIR = SFT_OUTPUT_DIR / ("sft_full_ckpt" if FULL_FT else "sft_adapter")
SFT_CKPT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
torch.manual_seed(SEED)
sft_device = torch.device("cuda" if HAS_GPU else "cpu")
mode_str = "全量微调" if FULL_FT else "LoRA 微调"
print(f"设备：{sft_device} | 模式：{mode_str} | 模型：{SFT_MODEL_NAME}")
print(f"epochs={SFT_EPOCHS} | batch={SFT_BATCH_SIZE} | max_len={SFT_MAX_LENGTH} | "
      f"train_limit={SFT_TRAIN_LIMIT or '全量'} | lr={SFT_LR}")
print(f"权重保存目录：{SFT_CKPT_DIR}")

# ── 1. 数据（原始记录）─────────────────────────────────────────────────────
_, _, sft_id2label = build_corpus_label_schema()
train_records = load_split_json("corpus", "train")
val_records = load_split_json("corpus", "validation")
if SFT_TRAIN_LIMIT and SFT_TRAIN_LIMIT > 0:
    train_records = train_records.select(range(min(SFT_TRAIN_LIMIT, len(train_records))))
if SFT_EVAL_LIMIT and SFT_EVAL_LIMIT > 0:
    val_records = val_records.select(range(min(SFT_EVAL_LIMIT, len(val_records))))
print(f"训练集：{len(train_records)} 条 | 验证集：{len(val_records)} 条")

_demo_text, _demo_target = record_to_text_and_target(train_records[0], sft_id2label)
print(f"\n样例文本：{_demo_text[:40]}...")
print(f"样例目标：{_demo_target}")

# ── 2. Tokenizer ─────────────────────────────────────────────────────────────
print(f"\n加载 tokenizer：{SFT_MODEL_NAME}")
sft_tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_NAME, trust_remote_code=True)
if sft_tokenizer.pad_token_id is None:
    sft_tokenizer.pad_token_id = sft_tokenizer.eos_token_id

# ── 3. 预 tokenize：把所有记录一次性转成 ID，之后 __getitem__ 只做切片 ──────
# 这一步在 CPU 上运行（即使有 GPU 也无需 GPU），完成后 tokenizer 不再参与训练循环。
print("\n预 tokenize 训练集（一次性，之后训练时无 tokenize 开销）...")
train_items = pretokenize_records(train_records, sft_tokenizer, sft_id2label)
print(f"  完成，{len(train_items)} 条")
print("预 tokenize 验证集...")
val_items = pretokenize_records(val_records, sft_tokenizer, sft_id2label)
print(f"  完成，{len(val_items)} 条")

_collate = lambda b: collate_fn(b, sft_tokenizer.pad_token_id)
sft_train_loader = DataLoader(SFTDataset(train_items, SFT_MAX_LENGTH),
                              batch_size=SFT_BATCH_SIZE, shuffle=True,
                              collate_fn=_collate)
sft_val_loader   = DataLoader(SFTDataset(val_items, SFT_MAX_LENGTH),
                              batch_size=SFT_BATCH_SIZE * 2, shuffle=False,
                              collate_fn=_collate)

# ── 4. 模型 + LoRA / 全量 ────────────────────────────────────────────────────
print(f"\n加载 base model：{SFT_MODEL_NAME}")
sft_model = AutoModelForCausalLM.from_pretrained(
    SFT_MODEL_NAME,
    dtype=torch.float32,        # transformers 5.x 用 dtype= 不用 torch_dtype=
    trust_remote_code=True,
)

if FULL_FT:
    total = sum(p.numel() for p in sft_model.parameters())
    print(f"trainable params: {total:,} || all params: {total:,} || trainable%: 100.0000")
else:
    if not PEFT_AVAILABLE:
        raise ImportError("LoRA 模式需要 peft 库：pip install peft>=0.14.0")
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
    )
    sft_model = get_peft_model(sft_model, lora_config)
    sft_model.print_trainable_parameters()

sft_model = sft_model.to(sft_device)

# ── 5. 优化器 ────────────────────────────────────────────────────────────────
sft_optimizer = AdamW(sft_model.parameters(), lr=SFT_LR, weight_decay=0.01)
total_steps = len(sft_train_loader) * SFT_EPOCHS // GRAD_ACCUM
print(f"总训练步数：{total_steps}（batch={SFT_BATCH_SIZE}, grad_accum={GRAD_ACCUM}, "
      f"epochs={SFT_EPOCHS}）\n")

# ── 6. 训练循环 ───────────────────────────────────────────────────────────────
best_val_loss = float("inf")
sft_log_records = []

for epoch in range(1, SFT_EPOCHS + 1):
    sft_model.train()
    total_loss, total_tokens = 0.0, 0
    sft_optimizer.zero_grad()
    t0 = time.time()

    pbar = tqdm(sft_train_loader, desc=f"Epoch {epoch}/{SFT_EPOCHS} [Train]", leave=False)
    for step, batch in enumerate(pbar):
        input_ids = batch["input_ids"].to(sft_device)
        attention_mask = batch["attention_mask"].to(sft_device)
        labels = batch["labels"].to(sft_device)

        outputs = sft_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        (loss / GRAD_ACCUM).backward()
        if (step + 1) % GRAD_ACCUM == 0:
            nn.utils.clip_grad_norm_(sft_model.parameters(), max_norm=1.0)
            sft_optimizer.step()
            sft_optimizer.zero_grad()

        n_tokens = (labels != -100).sum().item()
        total_loss += loss.item() * n_tokens
        total_tokens += n_tokens
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss = total_loss / max(total_tokens, 1)

    sft_model.eval()
    val_loss, val_tokens = 0.0, 0
    with torch.no_grad():
        for batch in tqdm(sft_val_loader, desc="Val", leave=False):
            input_ids = batch["input_ids"].to(sft_device)
            attention_mask = batch["attention_mask"].to(sft_device)
            labels = batch["labels"].to(sft_device)
            outputs = sft_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            n_tokens = (labels != -100).sum().item()
            val_loss += outputs.loss.item() * n_tokens
            val_tokens += n_tokens
    avg_val_loss = val_loss / max(val_tokens, 1)

    elapsed = time.time() - t0
    print(f"Epoch {epoch}/{SFT_EPOCHS} | train_loss={avg_train_loss:.4f}  "
          f"val_loss={avg_val_loss:.4f} | {elapsed:.0f}s")
    sft_log_records.append({
        "epoch": epoch, "train_loss": avg_train_loss,
        "val_loss": avg_val_loss, "elapsed_s": elapsed,
    })

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        sft_model.save_pretrained(SFT_CKPT_DIR)
        sft_tokenizer.save_pretrained(SFT_CKPT_DIR)
        ckpt_label = "完整模型" if FULL_FT else "LoRA adapter"
        print(f"  ✓ 最优{ckpt_label}已保存 → {SFT_CKPT_DIR}  (val_loss={avg_val_loss:.4f})")

# ── 7. 保存训练日志 ───────────────────────────────────────────────────────────
sft_log_tag = "full_ft" if FULL_FT else "sft"
sft_log_path = SFT_OUTPUT_DIR / "logs" / f"train_{sft_log_tag}.json"
sft_log_path.parent.mkdir(parents=True, exist_ok=True)
with open(sft_log_path, "w", encoding="utf-8") as f:
    json.dump(sft_log_records, f, ensure_ascii=False, indent=2)

ckpt_label = "完整模型" if FULL_FT else "LoRA adapter"
print(f"\n训练完成。最优 val_loss={best_val_loss:.4f}")
print(f"训练日志 → {sft_log_path}")
print(f"{ckpt_label} → {SFT_CKPT_DIR}")

设备：cuda | 模式：LoRA 微调 | 模型：Qwen/Qwen2-0.5B-Instruct
epochs=3 | batch=4 | max_len=256 | train_limit=全量 | lr=0.0002
权重保存目录：/content/drive/MyDrive/ner_outputs/sft_adapter


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

训练集：20865 条 | 验证集：300 条

样例文本：海钓比赛地点在厦门与金门之间的海域。...
样例目标：{"entities": [{"text": "厦门", "type": "LOC"}, {"text": "金门", "type": "LOC"}]}

加载 tokenizer：Qwen/Qwen2-0.5B-Instruct


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

...

Val:   0%|          | 0/38 [00:00<?, ?it/s]

Epoch 2/3 | train_loss=0.0219  val_loss=0.0272 | 3075s
  ✓ 最优LoRA adapter已保存 → /content/drive/MyDrive/ner_outputs/sft_adapter  (val_loss=0.0272)


Epoch 3/3 [Train]:   0%|          | 0/5217 [00:00<?, ?it/s]

Val:   0%|          | 0/38 [00:00<?, ?it/s]

Epoch 3/3 | train_loss=0.0132  val_loss=0.0235 | 3068s
  ✓ 最优LoRA adapter已保存 → /content/drive/MyDrive/ner_outputs/sft_adapter  (val_loss=0.0235)

训练完成。最优 val_loss=0.0235
训练日志 → /content/drive/MyDrive/ner_outputs/logs/train_sft.json
LoRA adapter → /content/drive/MyDrive/ner_outputs/sft_adapter


## SFT 模型评估：corpus 验证集 span-level F1

上面的推理 demo 只做了定性展示；这个 cell 做**定量评估**：在验证集采样 N 条，计算 span-level F1，并与 BERT+CRF 做对比（运行完 LLM API 那部分后还可以加 API 三路线横向对比）。

**评估流程**：`tokens` 拼文本 → 送入 SFT 模型生成 JSON → `re` 提取 `entities` → `text.find()` 近似定位 → 与 gold span 做集合交集 → 精确率 / 召回率 / F1。

> ⚠️ **标准差异**：BERT+CRF（seqeval）用精确 token 边界；SFT（这里）用 `text.find()` 近似，差 1~2% 属正常误差，不影响横向对比结论。

In [ ]:
# ============ SFT 模型定量评估（corpus 验证集，span-level F1）============
# 教学重点：
#   1. 生成式 NER 的评估方式：JSON 解析 → span F1（与 LLM API cell 完全一致的标准）
#   2. LoRA adapter 自动识别：目录含 adapter_config.json → LoRA，否则 → 全量
#   3. parse_fail 统计：JSON 解析失败数，反映模型遵循指令的能力
#   4. 与 BERT+CRF 的横向对比：生成式 vs 序列标注各有什么优劣
import re
import json
import random
import time

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


# ── gold span 提取（corpus：tokens+ner_tags → {(text, type, start, end)}）────
def sft_gold_spans(record: dict, id2label: dict) -> set:
    """从 corpus 记录解码出 gold span 集合。

    与 LLM API cell 的 gold_spans_from_record 逻辑完全一致，复用同一评估标准。
    end 为闭区间，与 text.find() 定位方式对齐。
    """
    tokens, tag_ids = record["tokens"], record["ner_tags"]
    spans = set()
    cur_type, start = None, None
    for i, tid in enumerate(tag_ids):
        tag = id2label.get(tid, "O")
        if tag.startswith("B-"):
            if cur_type is not None:
                spans.add(("".join(tokens[start:i]), cur_type, start, i - 1))
            cur_type, start = tag[2:], i
        elif tag.startswith("I-") and cur_type == tag[2:]:
            continue
        else:
            if cur_type is not None:
                spans.add(("".join(tokens[start:i]), cur_type, start, i - 1))
            cur_type, start = None, None
    if cur_type is not None:
        spans.add(("".join(tokens[start:]), cur_type, start, len(tokens) - 1))
    return spans


def sft_pred_spans(text: str, raw_output: str, valid_types: set) -> set:
    """从 SFT 生成的 JSON 里提取预测 span，用 text.find() 近似定位。"""
    json_match = re.search(r"\{.*\}", raw_output, re.DOTALL)
    if not json_match:
        return set()
    try:
        obj = json.loads(json_match.group())
    except json.JSONDecodeError:
        return set()
    entities = obj.get("entities", [])
    if not isinstance(entities, list):
        return set()
    spans = set()
    for ent in entities:
        if not isinstance(ent, dict):
            continue
        surface = str(ent.get("text", "")).strip()
        etype = str(ent.get("type", "")).strip()
        if not surface or etype not in valid_types:
            continue
        idx = text.find(surface)
        if idx == -1:
            continue
        spans.add((surface, etype, idx, idx + len(surface) - 1))
    return spans


def sft_span_f1(all_golds: list, all_preds: list) -> dict:
    """span 级 P / R / F1（与 LLM API cell 中的 compute_span_f1 完全相同）。"""
    tp = sum(len(g & p) for g, p in zip(all_golds, all_preds))
    pred_total = sum(len(p) for p in all_preds)
    gold_total = sum(len(g) for g in all_golds)
    p = tp / pred_total if pred_total else 0.0
    r = tp / gold_total if gold_total else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    return {"precision": p, "recall": r, "f1": f1,
            "tp": tp, "pred_total": pred_total, "gold_total": gold_total}


# ── 模型加载（自动判断 LoRA / 全量）────────────────────────────────────────────
def _load_sft_for_eval(ckpt_dir):
    """从 SFT_CKPT_DIR 加载模型，LoRA 路径走 PeftModel，全量路径直接加载。"""
    _device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    is_lora = (ckpt_dir / "adapter_config.json").exists()
    if is_lora:
        from peft import PeftModel
        _tok = AutoTokenizer.from_pretrained(str(ckpt_dir), trust_remote_code=True)
        _base = AutoModelForCausalLM.from_pretrained(
            SFT_MODEL_NAME, dtype=torch.float32, trust_remote_code=True
        )
        _mdl = PeftModel.from_pretrained(_base, str(ckpt_dir)).to(_device)
        print(f"已加载 LoRA adapter：{ckpt_dir}")
    else:
        _tok = AutoTokenizer.from_pretrained(str(ckpt_dir), trust_remote_code=True)
        _mdl = AutoModelForCausalLM.from_pretrained(
            str(ckpt_dir), dtype=torch.float32, trust_remote_code=True
        ).to(_device)
        print(f"已加载全量微调模型：{ckpt_dir}")
    if _tok.pad_token_id is None:
        _tok.pad_token_id = _tok.eos_token_id
    _mdl.eval()
    return _mdl, _tok, _device


@torch.no_grad()
def _sft_generate(text: str, mdl, tok, dev, max_new_tokens=128) -> str:
    """单条推理，返回模型生成的原始字符串。"""
    prompt = tok.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user",   "content": text}],
        tokenize=False, add_generation_prompt=True,
    )
    inputs = tok(prompt, return_tensors="pt").to(dev)
    gen = mdl.generate(
        **inputs, max_new_tokens=max_new_tokens,
        do_sample=False, pad_token_id=tok.pad_token_id,
    )
    new_ids = gen[0][inputs["input_ids"].shape[1]:]
    return tok.decode(new_ids, skip_special_tokens=True).strip()


# ══════════════════════════════════════════════════════════════════════════════
# 评估配置（直接改这里）
# ══════════════════════════════════════════════════════════════════════════════
SFT_EVAL_N = 50        # 采样条数（CPU 上每条生成约 5~15s，建议先用小值跑通）
SFT_EVAL_SEED = 42

# 输出目录：复用 SAVE_ROOT（Drive 永久保存）；没挂载则退回本地
try:
    _sft_log_dir = SAVE_ROOT / "logs"
except NameError:
    _sft_log_dir = Path.cwd() / "outputs" / "logs"

# 检查 checkpoint 是否存在
if not SFT_CKPT_DIR.exists():
    print(f"找不到 SFT checkpoint：{SFT_CKPT_DIR}")
    print("请先运行 SFT 训练 cell（FULL_FT=False → sft_adapter/，FULL_FT=True → sft_full_ckpt/）。")
else:
    # ── 1. 加载模型 ──────────────────────────────────────────────────────────
    # 优先复用内核里已有的 sft_model（训练完直接评估，不重复加载）
    if "sft_model" not in globals():
        eval_mdl, eval_tok, eval_dev = _load_sft_for_eval(SFT_CKPT_DIR)
    else:
        eval_mdl, eval_tok, eval_dev = sft_model, sft_tokenizer, sft_device
        print("复用已加载的 sft_model，无需重新加载。")

    # ── 2. 数据采样 ──────────────────────────────────────────────────────────
    _, _, eval_id2label = build_corpus_label_schema()
    valid_types = set(SFT_TYPE_CN.keys())   # {"PER", "ORG", "LOC"}

    all_val = list(load_split_json("corpus", "validation"))
    random.seed(SFT_EVAL_SEED)
    eval_samples = random.sample(all_val, min(SFT_EVAL_N, len(all_val)))
    print(f"评估样本：{len(eval_samples)} 条（corpus 验证集随机采样）\n")

    # ── 3. 逐条推理 ──────────────────────────────────────────────────────────
    all_golds, all_preds = [], []
    parse_fail = 0
    sft_eval_detail = []
    t0 = time.time()

    for i, record in enumerate(eval_samples, 1):
        text = "".join(record["tokens"])
        g_set = sft_gold_spans(record, eval_id2label)
        raw = _sft_generate(text, eval_mdl, eval_tok, eval_dev)
        p_set = sft_pred_spans(text, raw, valid_types)

        if not re.search(r"\{.*entities.*\}", raw, re.DOTALL):
            parse_fail += 1

        all_golds.append(g_set)
        all_preds.append(p_set)
        sft_eval_detail.append({
            "text": text,
            "gold": [{"text": s, "type": t} for s, t, *_ in g_set],
            "pred": [{"text": s, "type": t} for s, t, *_ in p_set],
            "raw_output": raw,
        })

        if i % 10 == 0 or i == len(eval_samples):
            print(f"  已推理 {i}/{len(eval_samples)} 条")

    elapsed = time.time() - t0
    metrics = sft_span_f1(all_golds, all_preds)

    # ── 4. 读取 BERT+CRF 的结果（如有）做对比 ───────────────────────────────
    bert_f1_str = "?"
    _bert_log = _sft_log_dir / "eval_linear_validation.json"  # 线性头
    _crf_log = _sft_log_dir / "eval_crf_validation.json"      # CRF
    for _log in [_crf_log, _bert_log]:
        if _log.exists():
            with open(_log, encoding="utf-8") as _f:
                _d = json.load(_f)
            bert_f1_str = f"{_d.get('f1', '?'):.4f}" if isinstance(_d.get('f1'), float) else str(_d.get('f1', '?'))
            break

    # ── 5. 打印结果 ──────────────────────────────────────────────────────────
    print(f"\n{'='*65}")
    print(f"SFT 模型 NER 评估（corpus 验证集，n={len(eval_samples)}）")
    print(f"{'='*65}")
    print(f"  Precision : {metrics['precision']:.4f}")
    print(f"  Recall    : {metrics['recall']:.4f}")
    print(f"  F1        : {metrics['f1']:.4f}")
    print(f"  JSON 解析失败：{parse_fail} 条 ({parse_fail/len(eval_samples)*100:.1f}%)")
    print(f"  总耗时：{elapsed:.1f}s，均值 {elapsed/len(eval_samples):.2f}s/条")

    mode_tag = "LoRA" if (SFT_CKPT_DIR / "adapter_config.json").exists() else "全量"
    print(f"""
三路线对比（corpus 验证集随机采样，seed={SFT_EVAL_SEED}）
  ┌──────────────────────────────────────────────┬──────────┐
  │ 方法                                         │  F1      │
  ├──────────────────────────────────────────────┼──────────┤
  │ BERT + CRF / Linear（seqeval，全量数据）      │ {bert_f1_str:<8} │
  │ Qwen2-0.5B SFT（{mode_tag}，{len(eval_samples)} 条采样）        │ {metrics['f1']:.4f}   │
  └──────────────────────────────────────────────┴──────────┘

评估标准差异说明：
  · BERT+CRF 使用 seqeval（精确 token 边界），标准更严。
  · SFT 使用 span F1（text.find() 近似定位），与 LLM API 评估完全一致，
    两者可以直接横向比较；与 BERT 相差 1~2% 属正常误差。

思考题：
  1. SFT（本地微调小模型）vs LLM API（大模型 zero/few-shot），谁 F1 更高？为什么？
  2. parse_fail 数量代表什么？训练充分后这个数会如何变化？
  3. BERT+CRF 能保证零非法 BIO 序列，生成式 NER 有这个保证吗？怎么处理？
""")

    # ── 6. 保存结果 ──────────────────────────────────────────────────────────
    _sft_log_dir.mkdir(parents=True, exist_ok=True)
    _out_path = _sft_log_dir / "eval_sft.json"
    with open(_out_path, "w", encoding="utf-8") as _f:
        json.dump({
            "metrics": {k: round(v, 6) if isinstance(v, float) else v
                        for k, v in metrics.items()},
            "n_samples": len(eval_samples),
            "parse_fail": parse_fail,
            "mode": mode_tag,
            "detail": sft_eval_detail,
        }, _f, ensure_ascii=False, indent=2)
    print(f"评估结果已保存 → {_out_path}")

复用已加载的 sft_model，无需重新加载。
评估样本：50 条（corpus 验证集随机采样）

  已推理 10/50 条
  已推理 20/50 条
  已推理 30/50 条
  已推理 40/50 条
  已推理 50/50 条

SFT 模型 NER 评估（corpus 验证集，n=50）
  Precision : 0.9062
  Recall    : 0.7945
  F1        : 0.8467
  JSON 解析失败：0 条 (0.0%)
  总耗时：61.4s，均值 1.23s/条

三路线对比（corpus 验证集随机采样，seed=42）
  ┌──────────────────────────────────────────────┬──────────┐
  │ 方法                                         │  F1      │
  ├──────────────────────────────────────────────┼──────────┤
  │ BERT + CRF / Linear（seqeval，全量数据）      │ 0.9542   │
  │ Qwen2-0.5B SFT（LoRA，50 条采样）        │ 0.8467   │
  └──────────────────────────────────────────────┴──────────┘

评估标准差异说明：
  · BERT+CRF 使用 seqeval（精确 token 边界），标准更严。
  · SFT 使用 span F1（text.find() 近似定位），与 LLM API 评估完全一致，
    两者可以直接横向比较；与 BERT 相差 1~2% 属正常误差。

思考题：
  1. SFT（本地微调小模型）vs LLM API（大模型 zero/few-shot），谁 F1 更高？为什么？
  2. parse_fail 数量代表什么？训练充分后这个数会如何变化？
  3. BERT+CRF 能保证零非法 BIO 序列，生成式 NER 有这个保证吗？怎么处理？

评估结果已保存 → /content/drive/MyDrive/ner_outputs/logs/ev

# LLM API 直接做 NER：zero-shot vs few-shot（人民日报 corpus）

第三条技术路线：**不训练**，直接调用大模型 API（这里用阿里云 DashScope 的 Qwen），靠 prompt 让它输出实体 JSON。

| | BERT 序列标注 | LLM SFT（LoRA） | LLM API（这里） |
|---|---|---|---|
| 训练 | 全参数微调 | 微调 ~0.2% 参数 | **零训练**，纯 prompt |
| 成本 | 一次训练 | 一次训练 | 每条都要调 API（计费） |
| 关键 | 标注对齐 | loss masking | **prompt 设计** |

**两种 prompt 对比：**
- **zero-shot**：只给任务描述，不给样例。
- **few-shot**：额外给 3 个标注示例，引导模型对齐输出格式。

**评估**：与前两条路线保持可比 —— 统一用 **span 级 F1**（实体文本+类型+位置都对才算对）。corpus 是 `tokens`+`ner_tags`，先解码出 gold 实体的字符位置；LLM 预测的实体则在原文里 `find` 出位置。

**使用前提**：需要 DashScope API Key。Colab 里建议存到左侧 🔑 Secrets（名为 `DASHSCOPE_API_KEY`），或在下方 cell 里直接填。只采样少量样本（默认 50 条）以控制费用。

In [ ]:
# ============ LLM API 做 NER：zero-shot vs few-shot 对比（人民日报 corpus 版）============
# 教学重点：
#   1. LLM 做 NER 的 prompt 设计：zero-shot（只给任务描述）vs few-shot（给 3 个示例）
#   2. 结构化输出解析（从模型输出里抠 JSON + 容错）
#   3. span 级 F1（与 BERT / SFT 两条路线保持可比）
#   4. 成本控制：只采样少量样本，不跑完整验证集
import os
import re
import json
import time
import random
from collections import defaultdict

from openai import OpenAI


# 人民日报只有 3 类实体（与 build_corpus_label_schema 一致）
LLM_TYPE_CN = {"PER": "人名", "ORG": "机构", "LOC": "地名"}
LLM_ENTITY_TYPES = list(LLM_TYPE_CN.keys())   # ["PER", "ORG", "LOC"]


# ── API Key 与 client ────────────────────────────────────────────────────────
def get_api_key() -> str:
    """依次尝试：Colab Secrets → 环境变量 → 下方手填。"""
    # 1. Colab 左侧 🔑 Secrets（推荐，不会泄露到代码里）
    try:
        from google.colab import userdata
        key = userdata.get("DASHSCOPE_API_KEY")
        if key:
            return key
    except Exception:
        pass
    # 2. 环境变量
    key = os.getenv("DASHSCOPE_API_KEY")
    if key:
        return key
    # 3. 兜底：在这里直接填（注意别提交到 git）
    return "sk-在这里填入你的DashScope_API_Key"


def build_client() -> OpenAI:
    """DashScope 兼容 OpenAI 协议，换 base_url 即可。"""
    api_key = get_api_key()
    if not api_key or api_key.startswith("sk-在这里"):
        raise EnvironmentError(
            "未找到 DASHSCOPE_API_KEY。请在 Colab 左侧 🔑 Secrets 添加，"
            "或在 get_api_key() 里直接填。"
        )
    return OpenAI(
        api_key=api_key,
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",

    )


# ── gold / pred span 提取（统一成 {(text, type, start, end)}）────────────────
def gold_spans_from_record(record: dict, id2label: dict) -> set:
    """从 corpus 记录（tokens+ner_tags）解码出 gold spans。

    与 BIO→实体合并同理，但额外记录字符起止位置（end 为闭区间，与 llm 输出对齐）。
    返回 {(surface, type, start, end)}。
    """
    tokens, tag_ids = record["tokens"], record["ner_tags"]
    spans = set()
    cur_type, start = None, None
    for i, tid in enumerate(tag_ids):
        tag = id2label.get(tid, "O")
        if tag.startswith("B-"):
            if cur_type is not None:
                spans.add(("".join(tokens[start:i]), cur_type, start, i - 1))
            cur_type, start = tag[2:], i
        elif tag.startswith("I-") and cur_type == tag[2:]:
            continue
        else:
            if cur_type is not None:
                spans.add(("".join(tokens[start:i]), cur_type, start, i - 1))
            cur_type, start = None, None
    if cur_type is not None:
        spans.add(("".join(tokens[start:]), cur_type, start, len(tokens) - 1))
    return spans


def pred_spans_from_response(text: str, response_text: str) -> set:
    """从 LLM 输出里解析 span，在原文中 find 出位置。返回 {(surface, type, start, end)}。"""
    json_match = re.search(r"\{.*\}", response_text, re.DOTALL)
    if not json_match:
        return set()
    try:
        obj = json.loads(json_match.group())
    except json.JSONDecodeError:
        return set()

    entities = obj.get("entities", [])
    if not isinstance(entities, list):
        return set()

    spans = set()
    for ent in entities:
        if not isinstance(ent, dict):
            continue
        surface = str(ent.get("text", "")).strip()
        etype = str(ent.get("type", "")).strip()
        if not surface or etype not in LLM_ENTITY_TYPES:
            continue
        idx = text.find(surface)          # 取第一次出现的位置
        if idx == -1:
            continue
        spans.add((surface, etype, idx, idx + len(surface) - 1))
    return spans


def compute_span_f1(all_golds: list, all_preds: list) -> dict:
    """span 级 精确率 / 召回率 / F1。"""
    tp = sum(len(g & p) for g, p in zip(all_golds, all_preds))
    pred_total = sum(len(p) for p in all_preds)
    gold_total = sum(len(g) for g in all_golds)
    p = tp / pred_total if pred_total else 0.0
    r = tp / gold_total if gold_total else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    return {"precision": p, "recall": r, "f1": f1,
            "tp": tp, "pred_total": pred_total, "gold_total": gold_total}


# ── prompt 设计：zero-shot vs few-shot ───────────────────────────────────────
LLM_SYSTEM_PROMPT = (
    "你是一个命名实体识别（NER）专家，专门处理中文文本。\n"
    "请从用户输入的文本中识别以下 3 类实体，并以 JSON 格式输出结果：\n"
    "- PER：人名\n"
    "- ORG：机构（公司、政府、组织等）\n"
    "- LOC：地名（国家、城市、地点等）\n\n"
    "输出格式（严格遵守，不要包含其他文字）：\n"
    '{"entities": [{"text": "实体文本", "type": "实体类型英文名"}, ...]}\n\n'
    '如果没有实体，输出：{"entities": []}'
)

# few-shot 示例：风格贴近人民日报语料
LLM_FEW_SHOT_EXAMPLES = [
    {
        "text": "中共中央总书记江泽民在北京会见了来访的外国友人。",
        "output": '{"entities": [{"text": "中共中央", "type": "ORG"}, {"text": "江泽民", "type": "PER"}, {"text": "北京", "type": "LOC"}]}',
    },
    {
        "text": "李鹏总理考察了上海宝钢集团。",
        "output": '{"entities": [{"text": "李鹏", "type": "PER"}, {"text": "上海", "type": "LOC"}, {"text": "宝钢集团", "type": "ORG"}]}',
    },
    {
        "text": "今天天气晴朗，适合外出散步。",
        "output": '{"entities": []}',
    },
]


def zero_shot_prompt(text: str) -> list:
    return [
        {"role": "system", "content": LLM_SYSTEM_PROMPT},
        {"role": "user", "content": text},
    ]


def few_shot_prompt(text: str) -> list:
    messages = [{"role": "system", "content": LLM_SYSTEM_PROMPT}]
    for ex in LLM_FEW_SHOT_EXAMPLES:
        messages.append({"role": "user", "content": ex["text"]})
        messages.append({"role": "assistant", "content": ex["output"]})
    messages.append({"role": "user", "content": text})
    return messages


def call_api(client: OpenAI, messages: list, model: str) -> str:
    """调用 LLM API，返回文本输出，带简单重试。"""
    for attempt in range(3):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=0.0,         # NER 要稳定，关掉随机性
                max_tokens=512,
            )
            return resp.choices[0].message.content or ""
        except Exception as e:
            if attempt < 2:
                time.sleep(2 ** attempt)
            else:
                print(f"  API 调用失败：{e}")
                return ""
    return ""


def sample_corpus_records(n: int, seed: int = 42) -> list:
    """从 corpus 验证集采样 n 条，尽量覆盖 PER/ORG/LOC 三类。"""
    _, _, id2label = build_corpus_label_schema()
    records = list(load_split_json("corpus", "validation"))

    random.seed(seed)
    # 按实体类型分层
    by_type = defaultdict(list)
    for r in records:
        types = {id2label.get(t, "O")[2:] for t in r["ner_tags"] if t != 0}
        for etype in types:
            by_type[etype].append(r)

    selected_ids, selected = set(), []
    per_type = max(1, n // len(LLM_ENTITY_TYPES))
    for etype in LLM_ENTITY_TYPES:
        candidates = [r for r in by_type[etype] if id(r) not in selected_ids]
        chosen = random.sample(candidates, min(per_type, len(candidates)))
        for r in chosen:
            if len(selected) < n and id(r) not in selected_ids:
                selected_ids.add(id(r))
                selected.append(r)

    # 补足到 n 条
    remaining = [r for r in records if id(r) not in selected_ids]
    random.shuffle(remaining)
    for r in remaining:
        if len(selected) >= n:
            break
        selected.append(r)
    return selected[:n]


# ══════════════════════════════════════════════════════════════════════════════
# 配置（notebook 直接改这里，无需命令行）
# ══════════════════════════════════════════════════════════════════════════════
LLM_N_SAMPLES = 50               # 采样条数（每条要调 2 次 API：zero + few，注意费用）
LLM_MODEL = "qwen3.6-flash"          # 也可换 qwen-max / qwen-turbo
LLM_SEED = 42

# 输出目录：复用挂载好的 SAVE_ROOT（Drive 永久保存）；没挂载则退回本地
try:
    LLM_OUTPUT_DIR = SAVE_ROOT
except NameError:
    LLM_OUTPUT_DIR = Path.cwd() / "outputs"
LLM_LOG_DIR = LLM_OUTPUT_DIR / "logs"

_, _, llm_id2label = build_corpus_label_schema()
client = build_client()
records = sample_corpus_records(LLM_N_SAMPLES, seed=LLM_SEED)
print(f"模型：{LLM_MODEL} | 采样 {len(records)} 条验证集样本（每条调 2 次 API）\n")

zs_golds, zs_preds, fs_golds, fs_preds = [], [], [], []
detail_records = []

for i, record in enumerate(records, 1):
    text = "".join(record["tokens"])
    gold = gold_spans_from_record(record, llm_id2label)

    zs_resp = call_api(client, zero_shot_prompt(text), LLM_MODEL)
    zs_pred = pred_spans_from_response(text, zs_resp)
    fs_resp = call_api(client, few_shot_prompt(text), LLM_MODEL)
    fs_pred = pred_spans_from_response(text, fs_resp)

    zs_golds.append(gold); zs_preds.append(zs_pred)
    fs_golds.append(gold); fs_preds.append(fs_pred)

    detail_records.append({
        "text": text,
        "gold": [{"text": s, "type": t} for s, t, _, _ in gold],
        "zero_shot": [{"text": s, "type": t} for s, t, _, _ in zs_pred],
        "few_shot": [{"text": s, "type": t} for s, t, _, _ in fs_pred],
    })

    if i % 10 == 0 or i == len(records):
        print(f"  已处理 {i}/{len(records)} 条")

zs_metrics = compute_span_f1(zs_golds, zs_preds)
fs_metrics = compute_span_f1(fs_golds, fs_preds)

print("\n" + "=" * 60)
print(f"LLM NER 对比结果（模型：{LLM_MODEL}，样本：{len(records)} 条）")
print("=" * 60)
print(f"{'方案':<18} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 50)
print(f"{'Zero-shot':<18} {zs_metrics['precision']:>10.4f} {zs_metrics['recall']:>10.4f} {zs_metrics['f1']:>10.4f}")
print(f"{'Few-shot (3例)':<16} {fs_metrics['precision']:>10.4f} {fs_metrics['recall']:>10.4f} {fs_metrics['f1']:>10.4f}")

# 保存结果到 Drive
LLM_LOG_DIR.mkdir(parents=True, exist_ok=True)
result = {
    "model": LLM_MODEL,
    "n_samples": len(records),
    "zero_shot": {k: round(v, 6) for k, v in zs_metrics.items()},
    "few_shot": {k: round(v, 6) for k, v in fs_metrics.items()},
    "detail": detail_records,
}
out_path = LLM_LOG_DIR / "eval_llm_api.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)
print(f"\nLLM 评估结果已保存 → {out_path}")

模型：qwen3.6-flash | 采样 50 条验证集样本（每条调 2 次 API）

  已处理 10/50 条
  已处理 20/50 条
  已处理 30/50 条
  已处理 40/50 条
  已处理 50/50 条

LLM NER 对比结果（模型：qwen3.6-flash，样本：50 条）
方案                  Precision     Recall         F1
--------------------------------------------------
Zero-shot              0.8905     0.7974     0.8414
Few-shot (3例)        0.8712     0.7516     0.8070

LLM 评估结果已保存 → /content/drive/MyDrive/ner_outputs/logs/eval_llm_api.json


# 五方案汇总对比

把本 notebook 所有评估日志一次性读取，打印统一对比表。

> **评估标准说明**（为什么不能直接横向比较所有数字）
>
> | 路线 | 评估标准 | 备注 |
> |---|---|---|
> | BERT + Linear / CRF | **seqeval**，精确 token 边界 | 标准最严 |
> | LLM SFT | **span F1**，`text.find()` 近似定位 | 略宽，与 LLM API 可直接比 |
> | LLM API zero/few-shot | **span F1**，`text.find()` 近似定位 | 略宽，与 SFT 可直接比 |
>
> SFT 与 LLM API 使用完全一致的标准，可直接横向比较；与 BERT 对比时差 1~2% 属正常误差。

In [13]:
import json
from pathlib import Path

# ---------- 日志目录 ----------
try:
    _cmp_log_dir = Path(SAVE_ROOT) / "logs"
except NameError:
    _cmp_log_dir = Path.cwd() / "outputs" / "logs"

# ---------- 读取单条日志 ----------
def _load_log(path: Path):
    if not path.exists():
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)

linear_log = _load_log(_cmp_log_dir / "eval_linear_validation.json")
crf_log    = _load_log(_cmp_log_dir / "eval_crf_validation.json")
sft_log    = _load_log(_cmp_log_dir / "eval_sft.json")
llm_log    = _load_log(_cmp_log_dir / "eval_llm_api.json")

# ---------- 从日志中提取 P / R / F1 ----------
def _prf(log) -> tuple:
    """返回 (precision_str, recall_str, f1_str)，找不到显示 '-'"""
    if not log:
        return "-", "-", "-"

    def _fmt(v) -> str:
        if isinstance(v, float):
            # 已是 0~1 小数 → 转百分比；>1 则已是百分比
            return f"{v * 100:.2f}%" if v <= 1.0 else f"{v:.2f}%"
        return str(v) if v is not None else "-"

    # seqeval 日志结构: {"overall_precision": ..., "overall_recall": ..., "overall_f1": ...}
    # span-F1 日志结构: {"precision": ..., "recall": ..., "f1": ...}
    p = log.get("overall_precision", log.get("precision"))
    r = log.get("overall_recall",    log.get("recall"))
    f = log.get("overall_f1",        log.get("f1"))
    # span-F1 日志把 metrics 嵌套在 "metrics" 子 dict 里（eval_sft.json）
    if p is None and "metrics" in log:
        m = log["metrics"]
        p, r, f = m.get("precision"), m.get("recall"), m.get("f1")
    return _fmt(p), _fmt(r), _fmt(f)

linear_prf = _prf(linear_log)
crf_prf    = _prf(crf_log)
sft_prf    = _prf(sft_log)

# LLM API 日志含 zero_shot / few_shot 两组
zero_prf = few_prf = ("-", "-", "-")
if llm_log:
    zero_prf = _prf(llm_log.get("zero_shot", {}))
    few_prf  = _prf(llm_log.get("few_shot",  {}))

# ---------- 打印汇总表 ----------
header = f"{'方案':<22} {'评估标准':<14} {'Precision':>10} {'Recall':>8} {'F1':>8}"
sep    = "-" * len(header)

rows = [
    ("BERT + Linear",     "seqeval (严格)", *linear_prf),
    ("BERT + CRF",        "seqeval (严格)", *crf_prf),
    ("LLM SFT (LoRA)",    "span F1",       *sft_prf),
    ("LLM API zero-shot", "span F1",       *zero_prf),
    ("LLM API few-shot",  "span F1",       *few_prf),
]

print(sep)
print(header)
print(sep)
for name, std, p, r, f in rows:
    missing = p == "-" and r == "-" and f == "-"
    flag = "  ← 未运行" if missing else ""
    print(f"{name:<22} {std:<14} {p:>10} {r:>8} {f:>8}{flag}")
print(sep)

# ---------- 简要结论 ----------
def _raw_f1(log) -> float | None:
    """取 log 里 F1 的原始 float 值（可能嵌套在 metrics 子 dict）。"""
    if not log:
        return None
    v = log.get("overall_f1", log.get("f1"))
    if v is None and "metrics" in log:
        v = log["metrics"].get("f1")
    return v if isinstance(v, float) else None

print()
print("【结论参考】")
cf = _raw_f1(crf_log)
lf = _raw_f1(linear_log)
if cf is not None and lf is not None:
    diff = (cf - lf) * 100
    direction = "高" if diff > 0 else "低"
    reason = "CRF 约束非法转移序列，在小数据集上提升显著" if diff > 0 else "差距极小或数据量充足时 Linear 已足够"
    print(f"  CRF vs Linear  : CRF {direction} {abs(diff):.2f}%  ({reason})")

sf = _raw_f1(sft_log)
ff = _raw_f1(llm_log.get("few_shot", {}) if llm_log else {})
zf = _raw_f1(llm_log.get("zero_shot", {}) if llm_log else {})
if sf is not None and ff is not None:
    diff = (sf - ff) * 100
    direction = "高" if diff > 0 else "低"
    reason = "微调后模型对领域标签更稳定" if diff > 0 else "few-shot 示例质量高时 API 可超 SFT"
    print(f"  SFT vs few-shot: SFT {direction} {abs(diff):.2f}%  ({reason})")
if zf is not None and ff is not None:
    diff = (ff - zf) * 100
    direction = "few-shot 高" if diff > 0 else "zero-shot 高"
    reason = "示例有效减少格式幻觉" if diff > 0 else "示例可能引入噪声或模型已强泛化"
    print(f"  few-shot vs zero-shot: {direction} {abs(diff):.2f}%  ({reason})")

print()
print("注：BERT 用 seqeval，SFT/API 用 span F1，评估标准不同，F1 数值不可直接大小比较。")

------------------------------------------------------------------
方案                     评估标准            Precision   Recall       F1
------------------------------------------------------------------
BERT + Linear          seqeval (严格)       95.06%   95.79%   95.42%
BERT + CRF             seqeval (严格)        2.16%    2.16%    2.16%
LLM SFT (LoRA)         span F1            90.62%   79.45%   84.67%
LLM API zero-shot      span F1            89.05%   79.74%   84.14%
LLM API few-shot       span F1            87.12%   75.16%   80.70%
------------------------------------------------------------------

【结论参考】
  CRF vs Linear  : CRF 低 93.26%  (差距极小或数据量充足时 Linear 已足够)
  SFT vs few-shot: SFT 高 3.97%  (微调后模型对领域标签更稳定)
  few-shot vs zero-shot: zero-shot 高 3.44%  (示例可能引入噪声或模型已强泛化)

注：BERT 用 seqeval，SFT/API 用 span F1，评估标准不同，F1 数值不可直接大小比较。
